# SWEAT tutorials: 

Validation of the Interpolation Model (Rsd)

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import shutil

import xarray as xr

In [ ]:
from sweat.api import run_window_time_series

In [ ]:
from extra.model_plots import (
    plot_daily_et_comparison,
    plot_et_time_comparison,
    plot_pixel_time_statistical_distribution,
    plot_roi_error_statistical_distribution,
    plot_spatial_distribution_error,
    plot_time_error_characteristic_pixels,
    plot_time_error_with_extra_variable,
    plot_time_evolution_error,
    plot_variable_evolution,
    print_correlation_errors_variable,
    print_daily_statistics,
    print_pixel_statistics,
    print_roi_statistics,
)
from extra.model_tools import find_directories

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

In [ ]:
def clean(ts):
    """
    Clean outputs
    """
    # Remove directory
    shutil.rmtree(ts, ignore_errors=True)
    # Recreate directory
    os.makedirs(ts, exist_ok=True)

In [ ]:
def create_et_copy_dir(
    input_dir: str,
    freq: float | None = None,
    cloud_probability: float | None = None,
) -> str:
    """
    Create a directory containing observed evapotranspiration products for interpolation,
    based on a given acquisition frequency and a probability of cloud-contaminated (invalid) dates.

    Parameters
    ----------
    input_dir : str
        Directory where the evapotranspiration and radiation products are stored.
    freq : float, optional
        Acquisition frequency in days.
    cloud_probability : float, optional
        Probability that an observation is invalid.

    Returns
    -------
    str
        Path to the new directory.
    """
    et_original_dir = os.path.join(input_dir, "et")
    et_copy_dir = os.path.join(input_dir, "et_copy")
    shutil.rmtree(et_copy_dir, ignore_errors=True)
    os.makedirs(et_copy_dir, exist_ok=True)
    files = os.listdir(et_original_dir)
    files.sort()
    if freq is not None:
        files_to_keep = [f for idx, f in enumerate(files) if idx % freq == 0]
        for f in files_to_keep:
            shutil.copy(
                os.path.join(et_original_dir, f), os.path.join(et_copy_dir, f)
            )
        files = files_to_keep
    if cloud_probability is not None:
        for f in files:
            if random.random() < cloud_probability:  # noqa: S311
                os.remove(os.path.join(et_copy_dir, f))
    return et_copy_dir

### Input Data

We obtained the observed evapotranspiration, daily radiation and extra ERA5-Land products using the **prepare_daily_et_timeseries** and **prepare_daily_radiation_timeseries functions** from **ET-DATASET**.

In [ ]:
input_dir = os.path.join(
    "data", "timeseries"
)

In [ ]:
time = xr.date_range("2024-03-01", "2024-03-31")
min_date = time[0]
max_date = time[-1]

Retrieving data directories from the input data directory

In [ ]:
dir_sd, dir_rad, dir_var = find_directories(input_dir)
print("Daily radiation directory: ", dir_rad)
print("Daily ET directory: ", dir_sd)
print("Extra variables directory: ", dir_var)

Selection of a portion of daily data to simulate acquisitions

In [ ]:
et_copy_dir = create_et_copy_dir(input_dir, freq=3)
print("Daily ET single dates directory: ", et_copy_dir)

### Output 

In [ ]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

In [ ]:
dir_ts = os.path.join(output_path, "et_timeseries")

### Compute ET time series

  We then generate the simulated evapotranspiration products using the **run_window_time_series** function from **SWEAT**.

In [ ]:
clean(dir_ts)
run_window_time_series(
    period_start=time[0],
    period_end=time[-1],
    et_single_date_dir=et_copy_dir,
    radiation_dir=dir_rad,
    et_time_series_dir=dir_ts,
    window=7,
    shift=1,
)

We run the model for the Coastal Region of Senegal in March 2024, benefiting from ideal meteorological conditions (no rain or clouds). 

### Single date statistics

In [ ]:
plot_daily_et_comparison(dir_sd, dir_ts, "2024-03-02")

In [ ]:
print_daily_statistics(dir_sd, dir_ts, "2024-03-02")

### Spatial distribution of errors computed over the entire period

In [ ]:
plot_spatial_distribution_error(dir_sd, dir_ts, min_date, max_date)

In [ ]:
print_roi_statistics(dir_sd, dir_ts, min_date, max_date)

In [ ]:
plot_roi_error_statistical_distribution(
    dir_sd, dir_ts, min_date, max_date, "mae"
)

### Time evolution of the error at a single pixel

In [ ]:
x = 394479
y = 1539972

In [ ]:
print_pixel_statistics(dir_sd, dir_ts, min_date, max_date, x, y)

In [ ]:
plot_et_time_comparison(dir_sd, dir_ts, min_date, max_date, x, y)

In [ ]:
plot_time_evolution_error(dir_sd, dir_ts, min_date, max_date, x, y)

In [ ]:
plot_pixel_time_statistical_distribution(
    dir_sd, dir_ts, min_date, max_date, x, y, absolute=True
)

In [ ]:
plot_time_error_characteristic_pixels(
    dir_sd, dir_ts, min_date, max_date, absolute=True
)

### Linking error to ERA5-Land variables

In [ ]:
plot_variable_evolution(dir_var, dir_sd, dir_rad, min_date, max_date, x, y)

In [ ]:
print_correlation_errors_variable(
    dir_var, dir_sd, dir_ts, "spearman", min_date, max_date, x, y, absolute=True
)

In [ ]:
print_correlation_errors_variable(
    dir_var, dir_sd, dir_ts, "pearson", min_date, max_date, x, y, absolute=True
)

In [ ]:
plot_time_error_with_extra_variable(
    dir_var, dir_sd, dir_ts, "sw", min_date, max_date, x, y, absolute=True
)